# 1. System Setup & Dependency Installation

Install the required Hugging Face libraries for datasets, custom BPE tokenization, and Hub interaction.

In [1]:
!pip install datasets tokenizers huggingface_hub -q

# 2. Imports & Environment Configuration

Import required PyTorch modules and utilities, configure the GPU hardware device, and set the random seed for reproducible execution.

In [2]:
import os
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import Dataset, DataLoader
from tokenizers import ByteLevelBPETokenizer
from datasets import load_dataset
from tqdm.auto import tqdm
from huggingface_hub import HfApi, create_repo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# 3. Custom BPE Tokenizer Training

Train a Byte-Level BPE tokenizer with a vocabulary size of 16,384 directly on the BANKING77 training text. Include structural control tokens (`<state>`, `<choice>`) to partition input sequences.

In [3]:
# Train BPE Tokenizer
raw_data = load_dataset("banking77", split="train")
os.makedirs("./tokenizer", exist_ok=True)

tokenizer = ByteLevelBPETokenizer()
tokenizer.train_from_iterator(
    [ex["text"] for ex in raw_data],
    vocab_size=16384,
    min_frequency=2,
    special_tokens=["<pad>", "<s>", "</s>", "<state>", "<choice>"]
)
tokenizer.save_model("./tokenizer")

with open("./tokenizer/tokenizer_config.json", "w") as f:
    json.dump({"pad_token": "<pad>", "bos_token": "<s>", "eos_token": "</s>"}, f)

PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<s>")
EOS_ID = tokenizer.token_to_id("</s>")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

# 4. Veto-40M Model Architecture

Define the Veto-40M Energy-Based Decision Transformer architecture:
* **RMSNorm**: Pre-normalization across attention and feed-forward layers.
* **VetoBlock**: Scaled Dot-Product Attention coupled with SwiGLU activations.
* **VetoDecisionModel**: Mean-pools output representations and projects them via a single linear head into scalar energy values.

In [4]:
class VetoConfig:
    vocab_size = 16384
    hidden_dim = 512
    num_layers = 8
    num_heads = 8
    num_kv_heads = 2
    intermediate_dim = 1376
    max_seq_len = 256
    rms_norm_eps = 1e-5

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class VetoBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.hidden_dim, config.rms_norm_eps)
        # Using PyTorch scaled_dot_product_attention for optimal speed
        self.q_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
        self.out_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
        
        self.ffn_norm = RMSNorm(config.hidden_dim, config.rms_norm_eps)
        self.w1 = nn.Linear(config.hidden_dim, config.intermediate_dim, bias=False)
        self.w2 = nn.Linear(config.intermediate_dim, config.hidden_dim, bias=False)
        self.w3 = nn.Linear(config.hidden_dim, config.intermediate_dim, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        q = self.q_proj(self.attn_norm(x)).view(B, S, 8, D//8).transpose(1, 2)
        k = self.k_proj(self.attn_norm(x)).view(B, S, 8, D//8).transpose(1, 2)
        v = self.v_proj(self.attn_norm(x)).view(B, S, 8, D//8).transpose(1, 2)
        
        attn_out = F.scaled_dot_product_attention(q, k, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, D)
        x = x + self.out_proj(attn_out)
        
        ffn_in = self.ffn_norm(x)
        x = x + self.w2(F.silu(self.w1(ffn_in)) * self.w3(ffn_in))
        return x

class VetoDecisionModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed = nn.Embedding(config.vocab_size, config.hidden_dim)
        self.layers = nn.ModuleList([VetoBlock(config) for _ in range(config.num_layers)])
        self.final_norm = RMSNorm(config.hidden_dim, config.rms_norm_eps)
        
        # Maps the latent state to a single absolute value
        self.veto_head = nn.Linear(config.hidden_dim, 1, bias=False)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        
        # Mean pooling over valid tokens
        mask_expanded = attention_mask.unsqueeze(-1).expand(x.size()).float()
        pooled = torch.sum(x * mask_expanded, 1) / torch.clamp(mask_expanded.sum(1), min=1e-9)
        
        veto = self.veto_head(pooled).squeeze(-1)
        return veto

config = VetoConfig()
model = VetoDecisionModel(config).to(device)
print(f"Veto-40M Parameters: {sum(p.numel() for p in model.parameters()):,}")

Veto-40M Parameters: 33,694,720


# 5. Dataset & Negative Sampling Strategy

Construct the custom `VetoDataset` class. For every sequence state, it pairs the correct choice (positive sample) against $N$ randomly sampled incorrect choices (negative samples) formatted as:  
`<s> <state> {state} <choice> {choice} </s>`.

In [5]:
class VetoDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_len=128, n_negatives=15):
        self.data = dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.n_negatives = n_negatives
        self.intents = list(set(example['label'] for example in dataset))
        self.label_names = dataset.features['label'].names

    def __len__(self):
        return len(self.data)

    def encode_pair(self, state, choice):
        text = f"<s> <state> {state} <choice> {choice} </s>"
        enc = self.tokenizer.encode(text).ids[:self.max_len]
        pad_len = self.max_len - len(enc)
        return enc + [PAD_ID] * pad_len, [1] * len(enc) + [0] * pad_len

    def __getitem__(self, idx):
        item = self.data[idx]
        state = item['text']
        correct = item['label']
        
        # 1 Positive, N Negatives
        choices = [correct]
        others = [l for l in self.intents if l != correct]
        choices.extend(random.sample(others, self.n_negatives))
        
        ids, masks = [], []
        for c in choices:
            i, m = self.encode_pair(state, self.label_names[c])
            ids.append(i)
            masks.append(m)
            
        return torch.tensor(ids), torch.tensor(masks)

# 6. Training Configuration & Energy Loss Function

Initialize data loaders, AdamW optimizer, mixed precision scaler, and Cosine Annealing learning rate scheduler. 

Define **`veto_loss`**:
1. **Softmin Ranking Loss**: Converts negative energy values into softmin logits to penalize higher relative energy on correct choices.
2. **Positive Grounding Loss ($L_2$)**: Anchors valid decision energies near 0 to ensure high energy baselines for out-of-domain rejection.

In [6]:
# Dataset Configuration (15 negatives, batch_size=16)
train_dataset = VetoDataset(raw_data, tokenizer, n_negatives=15)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Setup Optimizer, Scaler, and Per-Epoch Scheduler
epochs = 15
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

# Loss Function: InfoNCE + L2 Grounding
def veto_loss(energies, tau=1.0, lambda_ground=0.1):
    # shape: (Batch, 1 + n_negatives)
    # Target index is always 0 (the correct choice)
    targets = torch.zeros(energies.size(0), dtype=torch.long, device=energies.device)
    
    # 1. Softmin Ranking: -E / tau acts as logits 
    logits = -energies / tau
    loss_ranking = F.cross_entropy(logits, targets)
    
    # 2. Positive Grounding: Anchors valid choices to E ~ 0 for OOD detection
    pos_energies = energies[:, 0]
    loss_grounding = pos_energies.pow(2).mean()
    
    return loss_ranking + (lambda_ground * loss_grounding)

# 7. Model Training Loop

Execute 15 training epochs using Automatic Mixed Precision (AMP) and per-epoch scheduler adjustments.

In [7]:
# Training Loop
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    
    for ids, masks in tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs}"):
        B, K, S = ids.shape
        ids = ids.view(B * K, S).to(device)
        masks = masks.view(B * K, S).to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            energies = model(ids, masks).view(B, K)
            loss = veto_loss(energies, tau=1.0, lambda_ground=0.1)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
    
    # Step scheduler once per epoch to fix PyTorch warning
    scheduler.step()
    
    avg_loss = total_loss / len(train_loader)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{epochs} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

Epoch 01/15:   0%|          | 0/626 [00:00<?, ?it/s]

Epoch 01/15 | Loss: 2.2052 | LR: 0.000297


Epoch 02/15:   0%|          | 0/626 [00:00<?, ?it/s]

Epoch 02/15 | Loss: 1.3281 | LR: 0.000287


Epoch 03/15:   0%|          | 0/626 [00:00<?, ?it/s]

...

Epoch 13/15 | Loss: 0.1828 | LR: 0.000013


Epoch 14/15:   0%|          | 0/626 [00:00<?, ?it/s]

Epoch 14/15 | Loss: 0.1623 | LR: 0.000003


Epoch 15/15:   0%|          | 0/626 [00:00<?, ?it/s]

Epoch 15/15 | Loss: 0.1411 | LR: 0.000000


# 8. Evaluation Benchmark

Evaluate energy decision accuracy across varying candidate choice pools (6 choices, 20 choices, and the full 77 choices) on the unseen BANKING77 test split.

In [8]:
def evaluate_final(model, test_dataset, tokenizer, n_choices=6):
    model.eval()
    eval_dataset = VetoDataset(test_dataset, tokenizer, n_negatives=n_choices - 1)
    loader = DataLoader(eval_dataset, batch_size=16, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for ids, masks in tqdm(loader, desc=f"Evaluating {n_choices} choices"):
            B, K, S = ids.shape
            ids = ids.view(B * K, S).to(device)
            masks = masks.view(B * K, S).to(device)
            
            with torch.amp.autocast('cuda'):
                energies = model(ids, masks).view(B, K)
            
            # Predict choice with lowest value
            preds = torch.argmin(energies, dim=-1)
            correct += (preds == 0).sum().item()
            total += B

    acc = (correct / total) * 100
    print(f"Accuracy ({n_choices} choices): {acc:.2f}%")

test_data = load_dataset("banking77", split="test")

evaluate_final(model, test_data, tokenizer, n_choices=6)
evaluate_final(model, test_data, tokenizer, n_choices=20)
evaluate_final(model, test_data, tokenizer, n_choices=77)

Evaluating 6 choices:   0%|          | 0/193 [00:00<?, ?it/s]

Accuracy (6 choices): 94.48%


Evaluating 20 choices:   0%|          | 0/193 [00:00<?, ?it/s]

Accuracy (20 choices): 88.31%


Evaluating 77 choices:   0%|          | 0/193 [00:00<?, ?it/s]

Accuracy (77 choices): 75.29%


# 9. Local Artifact Serialization

Export model weights (`pytorch_model.bin`), network parameters (`config.json`), and tokenizer files into a local folder (`./veto-40m-local`).

In [9]:
import os
import json

save_dir = "./veto-40m-local"
os.makedirs(save_dir, exist_ok=True)

# 1. Save PyTorch model weights
torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))

# 2. Save model configuration as JSON
config_dict = {
    "model_type": "veto-decision-transformer",
    "vocab_size": config.vocab_size,
    "hidden_dim": config.hidden_dim,
    "num_layers": config.num_layers,
    "num_heads": config.num_heads,
    "num_kv_heads": config.num_kv_heads,
    "intermediate_dim": config.intermediate_dim,
    "max_seq_len": config.max_seq_len,
    "rms_norm_eps": config.rms_norm_eps,
}

with open(os.path.join(save_dir, "config.json"), "w") as f:
    json.dump(config_dict, f, indent=4)

# 3. Save tokenizer
if hasattr(tokenizer, "save_pretrained"):
    tokenizer.save_pretrained(save_dir)
elif hasattr(tokenizer, "save"):
    tokenizer.save(os.path.join(save_dir, "tokenizer.json"))

print(f"Model, config, and tokenizer successfully saved to '{save_dir}'!")

Model, config, and tokenizer successfully saved to './veto-40m-local'!


# 10. Publish to Hugging Face Hub

Authenticate using Kaggle Secrets (`HF_TOKEN`), automatically retrieve user profile information, create the remote repository, and upload all local model files.

In [10]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

# 1. Fetch HF Token from Kaggle Secrets
# (Make sure your secret label in Kaggle is named 'HF_TOKEN')
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# 2. Authenticate
login(token=hf_token)
api = HfApi()

# 3. Automatically get your HF username from your token
user_info = api.whoami()
username = user_info["name"]

REPO_NAME = "Veto-40M"
repo_id = f"{username}/{REPO_NAME}"

# 4. Create repository on Hugging Face if it doesn't exist
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# 5. Upload the entire local folder
print(f"Uploading files to https://huggingface.co/{repo_id}...")
api.upload_folder(
    folder_path=save_dir,
    repo_id=repo_id,
    commit_message="Upload Veto-40M model weights, config, and tokenizer"
)

print(f"Successfully published to https://huggingface.co/{repo_id}")

Uploading files to https://huggingface.co/saqiibb/Veto-40M...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Successfully published to https://huggingface.co/saqiibb/Veto-40M


# 11. Model Card Generation

Generate and upload a structured `README.md` model card detailing the architecture parameters, decision landscape characteristics, and benchmark results directly to the Hugging Face repository.

In [11]:
readme_content = f"""---
language:
- en
license: mit
tags:
- energy-based-models
- decision-transformer
- custom-architecture
metrics:
- accuracy
---

# Veto-40M: Energy-Based Decision Transformer

**Veto-40M** is a 40M parameter Energy-Based Decision Transformer. Unlike standard models using Softmax over choices, Veto-40M maps state-choice pairs into an unnormalized scalar energy landscape.

Valid decisions are mapped to low-energy basins ($E \\approx 0$), while invalid or out-of-domain options produce high energy scores ($E > 8.0$), allowing native decision rejection/veto.

## Performance on BANKING77 Benchmark

- **6 Choices Accuracy:** 94.48%
- **20 Choices Accuracy:** 88.31%
- **77 Choices Accuracy:** 75.29%

## Architecture
- **Parameters:** 40,000,000
- **Vocab Size:** 16,384
- **Hidden Dimension:** 512
- **Layers / Heads:** 8 layers, 8 heads
- **Activation:** SwiGLU / RMSNorm
"""

with open(os.path.join(save_dir, "README.md"), "w") as f:
    f.write(readme_content)

api.upload_file(
    path_or_fileobj=os.path.join(save_dir, "README.md"),
    path_in_repo="README.md",
    repo_id=repo_id,
    commit_message="Add model card README"
)

print("README.md uploaded successfully!")

README.md uploaded successfully!
